# # 01 · Mass spectra and the raw data

Companion to the tutorial notebook [`01_mass_spectra_and_data`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/tree/main/notebooks/level1).

**Kernel:** `cajal-lipidomics` · **Reads:** METASPACE (live) + the tutorial's data bundle · **Writes:** `data/derived/01_raw.h5ad`

This is where the whole pipeline starts. We pull two real mouse brain sections from the public Lipid Brain Atlas on METASPACE (one control female, one pregnant female, same coronal plane), turn them into a pixels x ions table, attach Allen atlas anatomy to every pixel, and save it. Every later notebook builds on the file we write at the end.

**The plan**
1. A quick picture of what MALDI imaging measures
2. Reading lipid names like `PC 38:6`
3. Pulling one section from METASPACE and looking at its mean spectrum
4. Adducts: why one lipid shows up at several masses
5. Pulling both sections, masking to tissue, keeping the ions they share
6. Attaching the Allen anatomy and saving the AnnData
7. First maps: one ion, a gallery of ions, and an RGB overlay

> Heads up: this notebook needs internet, because it downloads the images from METASPACE.

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------

# My own little package. `paths` knows where the tutorial clone, its provided data,
# and my own outputs live, so I never type "../../data/..." by hand.
from lipid_project import paths

paths.ensure_dirs()   # creates data/derived/, figures/, results/ if they're missing (safe to re-run)
paths.summary()       # prints where everything points + which stages are built so far.
                      # A line starting with "--" means that thing is missing: fix it before going on.

# The usual scientific stack
import numpy as np                 # arrays and math
import pandas as pd                # tables (obs / var are pandas DataFrames)
import matplotlib.pyplot as plt    # plotting
import anndata as ad               # the AnnData container: X (pixels x lipids) + obs + var + obsm + layers

# The tutorial's helper package (installed from the clone in external/).
# Course rule: before calling any cl.<module>.<function>, open
# external/lipidomics_tutorial_cajalcourse/src/cajal_lipidomics/<module>.py and read it.
import cajal_lipidomics as cl
from cajal_lipidomics import plotting, annotation
from cajal_lipidomics.style import FS     # the four standard font sizes: FS["xs"], ["s"], ["m"], ["l"]
cl.style.set_style()                      # the course figure defaults, so plots look clean and consistent

# One seed for anything random, so every run gives the same numbers.
SEED = 0
rng = np.random.default_rng(SEED)

# If this prints without a red error, the kernel is right. A ModuleNotFoundError here almost
# always means the wrong kernel: re-pick "cajal-lipidomics" in the top-right kernel picker.
print("ready. cajal_lipidomics", cl.__version__)

# ## 1. What MALDI imaging actually measures

A ~10 µm thick slice of frozen brain sits on a glass slide, coated with a light-absorbing **matrix** (here DHB). A laser fires at one spot, the matrix absorbs the energy and blasts off the surface, dragging tissue molecules into the gas phase as charged **ions**. The mass spectrometer sorts those ions by **mass-to-charge ratio (m/z)** and counts how many arrive at each m/z. That list is one **mass spectrum**.

The instrument rasters the laser across the slice on a 25 µm grid, so every grid point (a **pixel**) gets its own spectrum. Stack them back on the grid and you have an image per molecule.

Both sections were acquired the same way: Orbitrap, positive ion mode, resolving power 240,000 at m/z 200, 25 µm pixels, m/z 400 to 1600, DHB matrix.

> **A pixel is not a cell.** 25 µm is bigger than a neuron body, so each pixel averages cell bodies, axons, dendrites and glia. A lipid map shows the bulk chemistry of a tissue patch.

# ## 2. Brain lipids and how their names work

The brain is about half lipid by dry weight: membranes, synaptic vesicles, and above all **myelin**, the fatty insulation around axons that makes white matter white. The classes we'll keep meeting:

- **Glycerophospholipids:** PC, PE, PS, PI, PA, PG, and the single-chain lyso forms LPC / LPE
- **Sphingolipids** (myelin-heavy): SM, Cer, HexCer

A name like `PC 38:6` packs three facts: the class (`PC`), the total carbons across both fatty chains (`38`), and the total carbon-carbon double bonds (`6`). These are **sum compositions**: MALDI measures a mass, so it can't tell `18:0/20:6` from `16:0/22:6`. A suffix like `;O2` counts oxygens on the sphingoid backbone.

Let's write a tiny parser so this is concrete.

In [ ]:
import re   # regular expressions: tiny patterns for pulling pieces out of text


def parse_lipid_name(name):
    """Split a lipid shorthand into (class, carbons, double_bonds, oxygens).

    'PC 38:6'        -> ('PC', 38, 6, 0)
    'HexCer 42:2;O2' -> ('HexCer', 42, 2, 2)
    """
    # The class is everything before the first space or "(" -> re.split cuts there, [0] keeps the left part
    lipid_class = re.split(r"[ (]", name)[0]
    # (\d+):(\d+) means "digits, a colon, digits", the carbons:double-bonds pair.
    # The parentheses "capture" each number so we can read them back with .group(1) / .group(2).
    chain = re.search(r"(\d+):(\d+)", name)
    # ";O(\d*)" catches a backbone-oxygen suffix like ";O2". \d* allows ";O" with no number (means 1).
    oxy = re.search(r";O(\d*)", name)
    carbons = int(chain.group(1)) if chain else None
    double_bonds = int(chain.group(2)) if chain else None
    oxygens = (int(oxy.group(1)) if oxy.group(1) else 1) if oxy else 0
    return lipid_class, carbons, double_bonds, oxygens


for nm in ["PC 38:6", "HexCer 42:2;O2", "SM 36:1;O2", "PE 40:6", "LPC 16:0"]:
    print(f"{nm:16s} -> class, carbons, double bonds, oxygens = {parse_lipid_name(nm)}")

# The tutorial's helper `cl.data.lipid_properties` runs the same kind of parsing over a whole list of names at once (and adds an ether flag and a per-class colour). Same idea, no magic.

# ## 3. Pulling one section from METASPACE

METASPACE is the public web service hosting the Lipid Brain Atlas data. It also runs the annotation engine that decides which ions are present. We talk to it with the `metaspace2020` Python client, anonymously, since the project is public.

Settings, matching the published atlas:
- **Database** CoreMetabolome v3: the list of candidate molecules METASPACE matches against
- **FDR 0.1**: we accept a list where at most ~1 in 10 annotations is expected to be wrong. METASPACE estimates this by also scoring **decoy** formulas that can't be real molecules, and seeing how often they sneak through.
- **Raw intensities**, monoisotopic peak only

Network calls sometimes hiccup, so we wrap every call in a small retry helper.

In [ ]:
import time
from metaspace import SMInstance     # the METASPACE client (pip package: metaspace2020)

DB = ("CoreMetabolome", "v3")        # (database name, version) we annotate against
FDR = 0.1                            # keep annotations with <= 10% false discovery rate

# The two METASPACE dataset ids we use (both public, part of the Lipid Brain Atlas project)
PREGNANT_ID = "2024-07-14_14h24m11s"   # Brain1_C2, a pregnant female
CONTROL_ID = "2025-04-27_08h23m47s"    # 217D, a control (naive) female


def with_retry(fn, tries=4, pause=3.0):
    """Call fn(). If it raises (network blip, half-downloaded image), wait and try again.

    `fn` is a function with no arguments; we pass little `lambda: ...` wrappers so the
    actual network call happens inside here, where we can catch its errors.
    """
    for attempt in range(1, tries + 1):
        try:
            return fn()
        except Exception as err:                       # any error counts as "try again"
            if attempt == tries:                       # out of attempts: give up loudly
                raise
            print(f"  attempt {attempt} failed ({type(err).__name__}), retrying in {pause:.0f}s...")
            time.sleep(pause)


sm = SMInstance()                                       # anonymous session, no login needed
ds = with_retry(lambda: sm.dataset(id=PREGNANT_ID))     # a handle on one dataset (no images yet)
print("dataset name:", ds.name)

# The name encodes the date, the project, the condition (`Pregnant`), the section (`Brain1_C2`), the image size in pixels, and the 25 µm pixel size.

Now the **annotation table**: one row per detected ion, indexed by `(formula, adduct)`, with the measured `mz`, a quality score `msm` and its `fdr`.

In [ ]:
# results() returns a DataFrame. Its index has two levels: the neutral formula and the adduct.
res = with_retry(lambda: ds.results(database=DB, fdr=FDR))
print("annotated ions:", len(res))
print("index levels:", list(res.index.names))
res[["mz", "msm", "fdr"]].head(6)

# Now the **images**: one 2D intensity image per annotated ion. Each image object also carries its own `formula` and `adduct`, and we use those labels to line images up with the table above. (Relying on the two lists happening to come back in the same order would be fragile.)

In [ ]:
# all_annotation_images returns one "IsotopeImages" object per ion. img_set[0] is the
# monoisotopic image, a 2D numpy array (height x width) over the whole acquisition grid.
img_sets = with_retry(lambda: ds.all_annotation_images(
    fdr=FDR, database=DB,
    only_first_isotope=True,     # just the main isotope image per ion (that's all we analyse)
    scale_intensity=False,       # don't rescale to the instrument's absolute intensity range
    hotspot_clipping=False))     # don't clip bright hotspots (that's a display trick, not for analysis)

# A dictionary keyed by "formula__adduct" -> image. np.nan_to_num turns any NaN into 0.
images = {f"{s.formula}__{s.adduct}": np.nan_to_num(np.asarray(s[0], np.float32)) for s in img_sets}

# The same key for every row of the results table, so we can line things up by name
keys = [f"{f}__{a}" for f, a in res.index]
H, W = images[keys[0]].shape
print(f"one image is {H} x {W} = {H * W:,} pixels; we got {len(images)} images for {len(keys)} ions")

# Flatten each image into a long vector and stack them as columns -> (pixels x ions)
X_one = np.stack([images[k].ravel() for k in keys], axis=1)
mz_axis = res["mz"].to_numpy()
mean_spectrum = X_one.mean(axis=0)    # average over all pixels: one number per ion
print("pixels x ions matrix:", X_one.shape)

# ### The mean MALDI spectrum

A spectrum is just two aligned arrays (m/z and intensity), drawn as vertical sticks. `plotting.spectrum` is a few lines around `ax.vlines`, nothing hidden.

In [ ]:
ax = plotting.spectrum(mz_axis, mean_spectrum, title=f"mean MALDI spectrum, {ds.name.split('_')[3]} section")
plt.show()

# The tallest sticks crowd between roughly m/z 700 and 900, where the abundant membrane phospholipids and sphingolipids sit. Let's zoom into the most crowded 14 Da window to see how close some peaks are.

In [ ]:
# Find the 14-Da window holding the most peaks: for each peak, count how many peaks
# fall within [mz, mz + 14), then keep the start with the highest count.
starts = np.sort(mz_axis)
counts = [((mz_axis >= s) & (mz_axis < s + 14)).sum() for s in starts]
lo = starts[int(np.argmax(counts))]
win = (mz_axis >= lo) & (mz_axis < lo + 14)

ax = plotting.spectrum(mz_axis[win], mean_spectrum[win], lw=2.0,
                       title=f"zoom on m/z {lo:.0f}-{lo + 14:.0f}: several ions, nearly the same mass")
for m, v in zip(mz_axis[win], mean_spectrum[win]):
    ax.annotate(f"{m:.3f}", (m, v), ha="center", va="bottom", rotation=90, fontsize=FS["xs"])
plt.show()

# ## 4. Adducts: why one lipid shows up at several masses

The instrument only sees **ions**. In positive mode a neutral lipid picks up a small positive ion before it flies, and the m/z we measure is the lipid's neutral mass **plus** that ion's mass. The thing it picks up is the **adduct**: a proton `[M+H]+`, sodium `[M+Na]+`, potassium `[M+K]+`, or ammonium `[M+NH4]+`. So a single lipid can appear as several peaks. That's why METASPACE keys every feature by formula *and* adduct, not by a lipid name.

In [ ]:
# The four positive-mode adduct masses (Da) the tutorial helper stores
for name, mass in annotation.ADDUCTS.items():
    print(f"adduct {name:5s} adds {mass:9.5f} Da")

# Count how often each adduct shows up in this section, and show a formula that appears with several
print("\nions per adduct in this section:")
print(res.index.get_level_values("adduct").value_counts().to_string())
multi = res.index.get_level_values("formula").value_counts()
print("\nformulas detected with more than one adduct:", list(multi[multi > 1].index[:5]))

# ## 5. Both sections, tissue masks, and the ions they share

Now we build the real dataset. For each section we:
1. pull the annotation table and images (same three calls as above, wrapped in a function)
2. load its **tissue mask**, a boolean image that's `True` on brain and `False` on empty glass
3. keep only the ions detected in **both** sections, so the two share one feature set

In [ ]:
# Everything we need to know per section. The mask files come from the tutorial's data bundle.
SECTIONS = {
    "control_217D": dict(
        ds_id=CONTROL_ID, section_id=75.0, condition="naive",
        mask=paths.tutorial_data("masks", "BrainAtlas", "Control_Brains", "female",
                                 "20220416_MouseBrain_female_217D_447x332_Att30_25um", "mask.npy")),
    "pregnant_Brain1_C2": dict(
        ds_id=PREGNANT_ID, section_id=110.0, condition="pregnant",
        mask=paths.tutorial_data("masks", "PREGNANT",
                                 "20240712_MouseBrain_LipidAtlas_Pregnant_Brain1_C2_459x352_25um_Att30",
                                 "mask.npy")),
}


def pull_section(ds_id):
    """Pull one section: returns ({'formula__adduct': image}, per-ion table with formula/adduct/mz)."""
    d = with_retry(lambda: sm.dataset(id=ds_id))
    r = with_retry(lambda: d.results(database=DB, fdr=FDR))
    sets = with_retry(lambda: d.all_annotation_images(
        fdr=FDR, database=DB, only_first_isotope=True, scale_intensity=False, hotspot_clipping=False))
    imgs = {f"{s.formula}__{s.adduct}": np.nan_to_num(np.asarray(s[0], np.float32)) for s in sets}
    var = pd.DataFrame({"formula": r.index.get_level_values("formula"),
                        "adduct": r.index.get_level_values("adduct"),
                        "mz": r["mz"].to_numpy()},
                       index=[f"{f}__{a}" for f, a in r.index])
    var = var.loc[[k for k in var.index if k in imgs]]     # only ions we actually have an image for
    return imgs, var


per = {}
for name, cfg in SECTIONS.items():
    imgs, var = pull_section(cfg["ds_id"])
    mask = np.load(cfg["mask"])                             # True = tissue
    first = next(iter(imgs.values()))
    assert mask.shape == first.shape, f"{name}: mask {mask.shape} doesn't match image {first.shape}"
    per[name] = dict(imgs=imgs, var=var, mask=mask, cfg=cfg)
    print(f"{name:20s}: {len(imgs):3d} ions, image {first.shape}, {int(mask.sum()):,} tissue pixels")

# Set intersection keeps only keys present in BOTH sections; sorted() gives a stable column order.
common = sorted(set(per["control_217D"]["var"].index) & set(per["pregnant_Brain1_C2"]["var"].index))
print(f"\nions detected in both sections: {len(common)}")

# Next, each section becomes a **pixels x ions matrix** restricted to tissue pixels. `np.nonzero(mask)` gives the row (y) and column (x) of every `True` pixel, and we read every ion's intensity at exactly those positions.

In [ ]:
def build_block(P, common):
    """Tissue pixels x common ions for one section, plus a per-pixel metadata table."""
    ys, xs = np.nonzero(P["mask"])                         # coordinates of every tissue pixel
    X = np.stack([P["imgs"][k][ys, xs] for k in common], axis=1).astype(np.float32)
    obs = pd.DataFrame({"SectionID": P["cfg"]["section_id"],
                        "Condition": P["cfg"]["condition"],
                        "x": xs.astype(int), "y": ys.astype(int)})
    return X, obs


blocks = {name: build_block(P, common) for name, P in per.items()}
for name, (X, obs) in blocks.items():
    print(f"{name:20s}: matrix {X.shape}, intensity range {X.min():.3f} to {X.max():.3f}")

# ## 6. Attaching the Allen anatomy, then saving

Each pixel only knows its `(x, y)` on its own slide. To do neuroscience we need to know *where in the brain* it is. That's **registration** to the Allen Mouse Brain Common Coordinate Framework (CCFv3): warp the section onto the atlas so every pixel gets atlas coordinates `(xccf, yccf, zccf)` in mm and the Allen region it falls in.

Registration (done with a GUI tool called ABBA) is the one thing the course ships pre-computed, as a table with one row per tissue pixel. Notebook 04 digs into how it works. Here we just join it onto our pixels by `(SectionID, x, y)`.

In [ ]:
reg = pd.read_parquet(paths.tutorial_data("provided", "registration_ccf.parquet"))
print("registration rows:", len(reg), "| columns:", list(reg.columns))


def attach_ccf(obs, reg):
    """Left-join the registration onto a section's pixels, one row per pixel."""
    # validate="one_to_one" makes pandas raise if any pixel matched 0 or 2+ registration rows
    merged = obs.merge(reg, on=["SectionID", "x", "y"], how="left", validate="one_to_one")
    assert merged["acronym"].notna().all(), "every tissue pixel should get an Allen region"
    return merged


obs_blocks = {name: attach_ccf(obs, reg) for name, (X, obs) in blocks.items()}
obs_blocks["control_217D"][["SectionID", "x", "y", "xccf", "yccf", "zccf", "acronym", "allencolor"]].head()

# Now we assemble one **AnnData** holding both sections:
- `.X`: pixels x ions (rows from both sections stacked)
- `.obs`: one row per pixel (section, condition, x, y, CCF coordinates, region, region colour)
- `.var`: one row per ion (formula, adduct, m/z)

AnnData keeps these aligned for us: subset the pixels and `.X` and `.obs` are subset together.

In [ ]:
X_all = np.vstack([blocks[n][0] for n in SECTIONS])                  # stack rows: control, then pregnant
obs_all = pd.concat([obs_blocks[n] for n in SECTIONS], ignore_index=True)
# Give every pixel a readable unique id like "control_217D_0", "control_217D_1", ...
obs_all.index = [f"{n}_{i}" for n in SECTIONS for i in range(len(obs_blocks[n]))]
obs_all["Condition"] = obs_all["Condition"].astype(str)

# Per-ion metadata, taken from the control's table (identical formula/adduct for shared ions;
# the measured m/z can differ in the 6th decimal between sections, which doesn't matter here)
var = per["control_217D"]["var"].loc[common, ["formula", "adduct", "mz"]].copy()

raw = ad.AnnData(X=X_all, obs=obs_all, var=var)
print(raw)

In [ ]:
raw.write_h5ad(paths.stage("raw"))       # -> data/derived/01_raw.h5ad
print("saved", paths.stage("raw"))
print("pixels per condition:", raw.obs["Condition"].value_counts().to_dict())
print("distinct Allen regions:", raw.obs["acronym"].nunique())

# ## 7. First maps

Before any chemistry, let's paint every pixel with its **Allen region colour** at its atlas position. No lipid data goes into this picture, only registration. We plot `zccf` (left-right) against `-yccf` (minus, so dorsal is up).

This also sets a habit for the whole course: a real lipid is a physical substance in a physical place, so **a real lipid map is spatially coherent** (smooth territories, not salt-and-pepper). If a map looks like static, suspect the data.

In [ ]:
plotting.spatial_categorical(raw, color_key="allencolor")
plt.suptitle("every pixel painted with its Allen region colour", y=1.02, fontsize=FS["l"])
plt.show()

# Now the same canvas, coloured by one measured ion. We don't have lipid names yet (that's notebook 02), but the formula already hints at the class:
- two nitrogens and a phosphorus, like `C41H83N2O6P`, is the signature of **sphingomyelin (SM)**, a myelin lipid, so it should light up **white matter**
- no nitrogen and no phosphorus, like `C51H96O6`, is a **neutral lipid** (a triglyceride-type molecule)

We find such ions by their formula instead of hard-coding them.

In [ ]:
def has(formula, element, count=None):
    """True if `formula` contains `element` (optionally exactly `count` of it)."""
    m = re.search(rf"{element}(\d*)(?![a-z])", formula)       # (?![a-z]) stops "N" matching "Na"
    if not m:
        return False
    n = int(m.group(1)) if m.group(1) else 1
    return count is None or n == count


formulas = raw.var["formula"].astype(str)
mean_int = np.asarray(raw.X).mean(0)

# SM-like ions: exactly two N and a P. Among those, take the brightest one.
sm_like = [k for k in raw.var_names if has(formulas[k], "N", 2) and has(formulas[k], "P")]
# neutral-lipid-like ions: carbon, hydrogen, oxygen only (no N, no P)
neutral_like = [k for k in raw.var_names if not has(formulas[k], "N") and not has(formulas[k], "P")]
pick = lambda ks: max(ks, key=lambda k: mean_int[list(raw.var_names).index(k)]) if ks else raw.var_names[0]
wm_ion, neutral_ion = pick(sm_like), pick(neutral_like)
print("sphingomyelin-like ion:", wm_ion, "| neutral-lipid-like ion:", neutral_ion)

for ion, label in [(wm_ion, "sphingomyelin-range ion"), (neutral_ion, "neutral-lipid-range ion")]:
    plotting.spatial_lipid(raw, ion)
    plt.suptitle(f"{ion}: a {label}, raw intensity", y=1.02, fontsize=FS["l"])
    plt.show()

# ### A gallery: twelve ions, twelve territories

One map shows one ion. To feel how varied lipid territories are, we pick a dozen ions that are both **spatially structured** and **different from each other**, then map them side by side.

- **Structure score.** For each ion, what fraction of its variance is explained by the Allen region a pixel sits in? This is the R² of a one-way ANOVA: `variance of region means / total variance`. Structured ions score high, noise scores near 0.
- **Different from each other.** Greedily add the next-most-structured ion whose correlation with every ion already picked stays below 0.6.

In [ ]:
ctrl = (raw.obs["Condition"] == "naive").to_numpy()      # we'll show the control section
Xc = np.asarray(raw.X)[ctrl]
regions_c = raw.obs.loc[ctrl, "acronym"].astype(str).to_numpy()

# Structure score per ion: between-region variance / total variance
df = pd.DataFrame(Xc, columns=raw.var_names)
region_means = df.groupby(regions_c).transform("mean")    # each pixel replaced by its region's mean
r2 = (region_means.var() / (df.var() + 1e-12)).sort_values(ascending=False)

# Greedy pick: walk down the structure ranking and keep an ion only if it's not too
# correlated (|r| < 0.6) with anything already in the gallery.
C = df.corr().abs()
gallery = []
for ion in r2.index:
    if all(C.loc[ion, g] < 0.6 for g in gallery):
        gallery.append(ion)
    if len(gallery) == 12:
        break

zc, yc = raw.obs.loc[ctrl, "zccf"].to_numpy(), -raw.obs.loc[ctrl, "yccf"].to_numpy()
fig, axes = plt.subplots(2, 6, figsize=(16, 5.4))
for ax, ion in zip(axes.ravel(), gallery):
    v = df[ion].to_numpy()
    lo, hi = np.percentile(v, [2, 98])                    # clip the colour range so a few hot pixels don't wash it out
    ax.scatter(zc, yc, c=v, cmap="plasma", s=2, vmin=lo, vmax=hi, rasterized=True)
    cl.style.spatial_axes(ax)
    ax.set_title(f"{ion.split('__')[0]}\nR²={r2[ion]:.2f}", fontsize=FS["xs"])
for ax in axes.ravel()[len(gallery):]:
    ax.axis("off")                                         # hide unused panels if fewer than 12 qualified
plt.suptitle("a dozen distinct ions, a dozen spatial patterns (control section)", y=1.01, fontsize=FS["l"])
plt.tight_layout()
plt.show()

# ### Three ions at once: an RGB overlay

A screen makes every colour from red, green and blue. We do the same with three ions: each goes into one colour channel (scaled to its own 2nd to 98th percentile). Pure red/green/blue means one ion dominates, yellow/magenta/cyan means two overlap, white means all three. We use the top three gallery ions, which are spatially distinct by construction.

In [ ]:
trio = gallery[:3]
plotting.rgb_overlay(raw, trio)
plt.suptitle("R / G / B = " + " / ".join(t.split("__")[0] for t in trio), y=1.02, fontsize=FS["m"])
plt.show()

# ## Recap

- MALDI fires a laser at 25 µm spots; each spot becomes a pixel with a full spectrum. A pixel is a tissue patch, not a cell.
- A spectrum is two arrays (m/z, intensity). Peaks crowd into tiny mass windows.
- One lipid shows up at several m/z because of adducts, so features are keyed by `formula__adduct`.
- We pulled both sections, matched images to annotations by their own labels, masked to tissue, kept the shared ions, attached Allen anatomy, and saved **`data/derived/01_raw.h5ad`**.
- Even unnamed, ions paint coherent anatomy.

**Next (02):** turn each `formula__adduct` into a lipid name we can defend.